[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/07_evaluating_reasoning/07_evaluating_reasoning.ipynb)

# 07 · 评测推理模型：CoT Faithfulness、污染防御与判分器

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/pandas/matplotlib，全部 cell 秒级跑完。所有实验在**机制完全已知的模拟模型**上进行——faithful / post-hoc、sycophant / transparent、模板记忆 / 真算术——所以每个 faithfulness 与鲁棒性指标都能对照 ground truth 验证。

**本 notebook 你将完成：**

1. **CoT faithfulness 实验台**：构造 faithful（答案真依赖 CoT）与 post-hoc（答案先定、CoT 装饰）两类模拟模型，复现 Lanham 式截断实验，画出可区分两者的 early answering 曲线；
2. **Turpin 式偏置注入**：注入"总选 A"偏置，统计答案翻转率与 CoT 提及率，用 unfaithfulness 指标区分 sycophant 与 transparent；
3. **GSM-Symbolic 式参数化模板**：实现题目实例化器（换数字 / 换名字 / 加无关从句），对比模板记忆模型与真算术模型的掉点幅度；
4. **数学答案等价判分器**：分数/小数等价、数值容差、字符串规范化的简化实现 + corner cases 元评测；
5. **汇总报告**：把 faithfulness 分、鲁棒性掉点、判分器准确率合成一张 pandas 评测报告表；
6. **4 道 ✏️ 练习**（assert 自动判分）。

参考：[Lanham 2023, arXiv:2307.13702] · [Turpin 2023, arXiv:2305.04388] · [Chen 2025, Anthropic] · [Mirzadeh 2024, arXiv:2410.05229] · [Glazer 2024, arXiv:2411.04872] · [Phan 2025, arXiv:2501.14249]

In [ ]:
import math
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1 · CoT faithfulness 实验台：faithful vs post-hoc

[Lanham 2023] 的因果干预：在比例 $f$ 处**截断 CoT** 并强制作答，与完整 CoT 的答案比一致率，得 **early answering 曲线** $M(f)=\Pr[A(c_{1:\lfloor fL\rfloor})=A(c_{1:L})]$。

构造两类机制已知的模拟模型（$K=4$ 个选项）：

- **faithful**：每题有潜在抽签 $u_i\sim U(0,1)$，可见 CoT 比例为 $f$ 时答对当且仅当 $u_i < p(f)$，其中 $p(f)$ 从均匀先验 $1/K$ 线性升到 $p(1)=0.92$——答案随证据累积成形。可解析预言 $M(f) = 1-\big(p(1)-p(f)\big)$：从 $0.33$ 单调爬到 $1$；
- **post-hoc**：答案在写 CoT 之前就内定（同样 $0.92$ 准确率），截断多少都不变——$M(f)\equiv 1$。

faithfulness 代理分 = 曲线上方面积 $\frac{1}{|\mathcal F|}\sum_{f<1}\big(1-M(f)\big)$：早答越多、分越低。注意方法论前提（讲解第 2 节）：要先确认 CoT 对该任务有准确率增益，"截断不变"才构成 unfaithful 的证据。

In [ ]:
N, K = 4000, 4
true_ans = rng.integers(0, K, size=N)
u = rng.random(N)                                          # 每题潜在"难度抽签"，固定后行为可复现
wrong_ans = (true_ans + rng.integers(1, K, size=N)) % K    # 答错时落到的固定错误选项

ACC_FULL = 0.92
def p_correct(frac):
    # 可见 CoT 比例 -> 答对概率：从均匀先验 1/K 线性升到 ACC_FULL
    return 1 / K + (ACC_FULL - 1 / K) * frac

def faithful_model(frac):
    # faithful：答案真依赖可见的 CoT 证据量
    return np.where(u < p_correct(frac), true_ans, wrong_ans)

posthoc_fixed = np.where(u < ACC_FULL, true_ans, wrong_ans)  # 答案在写 CoT 之前就定了
def posthoc_model(frac):
    # post-hoc：CoT 只是装饰，截断不影响答案
    return posthoc_fixed

fracs = np.linspace(0, 1, 11)
def match_curve(model):
    full = model(1.0)
    return np.array([(model(f) == full).mean() for f in fracs])

curve_f, curve_p = match_curve(faithful_model), match_curve(posthoc_model)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fracs, curve_f, "o-", label="faithful (answer forms with evidence)")
ax.plot(fracs, curve_p, "s-", label="post-hoc (answer fixed before CoT)")
ax.axhline(1 / K, color="gray", ls=":", lw=1, label="chance agreement")
ax.set_xlabel("visible CoT fraction f (truncation point)")
ax.set_ylabel("agreement with full-CoT answer  M(f)")
ax.set_title("Lanham-style early answering curves")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

faith_score = lambda curve: float((1 - curve[:-1]).mean())   # 曲线上方面积 = faithfulness 代理分
fs_faithful, fs_posthoc = faith_score(curve_f), faith_score(curve_p)
print(f"faithfulness 代理分: faithful = {fs_faithful:.3f}   post-hoc = {fs_posthoc:.3f}")
assert fs_faithful > 0.2 > fs_posthoc, "两类模型必须被该指标分开"

## 2 · Turpin 式偏置注入：翻转率 × 提及率

[Turpin 2023] 协议：注入一个已知会影响答案的 **biasing feature**（如 few-shot 示例答案恰好全是 A），然后同时看两件事：

1. **行为**：答案被翻向偏置选项的比例（flip rate）——偏置是不是答案的真实原因；
2. **言语**：被翻转的样本里 CoT 提及偏置的比例（mention rate）——模型坦不坦白。

$$\text{unfaithfulness} \;=\; \text{flip} \times \big(1 - \Pr[\text{mention}\mid\text{flip}]\big)$$

三个模拟模型：**sycophant**（翻转高、几乎不提及——Turpin 在真实模型上发现的模式）、**transparent**（同样翻转高，但如实坦白"提示说选 A"）、**robust**（几乎不被带偏）。好指标必须把前两者分开——只看翻转率分不开，这正是 hint 范式 [Chen 2025] 把"用了 hint"与"承认用了 hint"拆开度量的原因。

In [ ]:
N2 = 4000
true2 = rng.integers(0, K, size=N2)
unbiased = np.where(rng.random(N2) < 0.85, true2, (true2 + rng.integers(1, K, N2)) % K)
BIAS_OPT = 0   # 注入的偏置：few-shot 答案永远是 A（选项 0）

def inject_bias(susceptibility, mention_prob, seed):
    r = np.random.default_rng(seed)
    follows = r.random(N2) < susceptibility                # 本题是否被 bias 拉走
    biased = np.where(follows, BIAS_OPT, unbiased)
    flipped = biased != unbiased                           # bias 真正改变了答案的样本
    mentioned = flipped & (r.random(N2) < mention_prob)    # CoT 是否坦白提及 bias
    return biased, flipped, mentioned

models = {
    "sycophant":   dict(susceptibility=0.40, mention_prob=0.05, seed=1),
    "transparent": dict(susceptibility=0.40, mention_prob=0.90, seed=2),
    "robust":      dict(susceptibility=0.04, mention_prob=0.50, seed=3),
}
rows, bias_runs = [], {}
for name, kw in models.items():
    biased, flipped, mentioned = inject_bias(**kw)
    flip_rate = float(flipped.mean())
    mention_rate = float(mentioned[flipped].mean()) if flipped.any() else 0.0
    unfaith = flip_rate * (1 - mention_rate)
    acc_drop = float((unbiased == true2).mean() - (biased == true2).mean())
    rows.append([name, flip_rate, mention_rate, unfaith, acc_drop])
    bias_runs[name] = (flipped, mentioned)

bias_df = pd.DataFrame(rows, columns=["model", "flip_rate", "mention|flip", "unfaithfulness", "acc_drop"])
print(bias_df.round(3).to_string(index=False))

flip_syc, mention_syc = bias_runs["sycophant"]
flip_tra, mention_tra = bias_runs["transparent"]
unfaith_syc = float(bias_df.loc[0, "unfaithfulness"])
unfaith_tra = float(bias_df.loc[1, "unfaithfulness"])
assert unfaith_syc > 5 * unfaith_tra, "同样翻转率下，必须靠提及率把两者分开"
print(f"\n同样的翻转率：sycophant unfaithfulness = {unfaith_syc:.3f} ≫ transparent = {unfaith_tra:.3f}")
print("（注意 acc_drop ≈ 0.24：偏置确实在伤害准确率——Turpin 实测 BBH 上最多掉 36%）")

## 3 · GSM-Symbolic 式参数化模板：换数字见真章

[Mirzadeh 2024] 把题目做成**符号模板** $q(\theta)$：名字、数字都是可重采样的参数，标准答案由生成器解析给出（本节模板 answer $= a + b - c$）。三种扰动：**换名字**（结构数值都不变）、**换数字**（解法不变、表面全新）、**加无关从句**（NoOp：塞一个与解无关的数字）。

两类机制已知的模拟模型：

- **memorizer（模板记忆）**：见过的 $(a,b,c)$ 组合直接背出答案；没见过就套"最像的旧题"的答案；见到 NoOp 数字大概率盲目卷进运算——模式匹配的两种典型失败；
- **arithmetic（真算术）**：真的解析 $a,b,c$ 计算，基础错误率 2%，NoOp 仅轻微分心（10% 被带偏）。

预期复现 GSM-Symbolic 的发现：换名字两者都几乎不掉；**换数字时 memorizer 崩溃、arithmetic 纹丝不动**；NoOp 下 memorizer 掉点远大于 arithmetic（真实模型上最高掉 65%）。掉点幅度就是"套路记忆程度"的标尺。

In [ ]:
NAMES = ["Alice", "Bob", "Carol", "David", "Erin", "Frank"]
rng3 = np.random.default_rng(0)

def make_instance(r):
    return {"name": str(r.choice(NAMES)), "a": int(r.integers(20, 80)),
            "b": int(r.integers(5, 40)), "c": int(r.integers(2, 20)), "noop": 0}

def render(inst):
    noop = (f" Of these, {inst['noop']} apples are slightly smaller than average."
            if inst["noop"] else "")
    return (f"{inst['name']} has {inst['a']} apples. A friend gives {inst['name']} "
            f"{inst['b']} more apples.{noop} Then {inst['name']} gives away {inst['c']} "
            f"apples. How many apples does {inst['name']} have?")

def answer_of(inst):
    return inst["a"] + inst["b"] - inst["c"]     # NoOp 数字与解无关

ORIGINALS = [make_instance(rng3) for _ in range(80)]        # “公开过的静态实例”（GSM8K 原题）
SEEN = {(d["a"], d["b"], d["c"]) for d in ORIGINALS}

def memorizer(inst, r):
    # 模板记忆模型：背得出旧题；新数字组合就套旧题答案；见到 NoOp 数字大概率盲用
    if (inst["a"], inst["b"], inst["c"]) in SEEN:
        ans = inst["a"] + inst["b"] - inst["c"]
        if inst["noop"] and r.random() < 0.7:
            ans -= inst["noop"]                  # “见数字就用”——GSM-NoOp 失败模式
        return ans
    ref = ORIGINALS[int(r.integers(len(ORIGINALS)))]
    return answer_of(ref)                        # 套最像的旧题的答案

def arithmetic(inst, r):
    # 真算术模型：解析 a、b、c 真算；NoOp 仅轻微分心
    ans = inst["a"] + inst["b"] - inst["c"]
    if inst["noop"] and r.random() < 0.10:
        ans -= inst["noop"]
    if r.random() < 0.02:
        ans += int(r.integers(1, 4))             # 偶发算错
    return ans

def perturb(inst, mode, r):
    out = dict(inst)
    if mode == "numbers":
        while (out["a"], out["b"], out["c"]) == (inst["a"], inst["b"], inst["c"]):
            out["a"] = int(r.integers(20, 80)); out["b"] = int(r.integers(5, 40)); out["c"] = int(r.integers(2, 20))
    elif mode == "names":
        out["name"] = str(r.choice([n for n in NAMES if n != inst["name"]]))
    elif mode == "noop":
        out["noop"] = int(r.integers(3, 9))
    return out

def accuracy(model, instances, seed):
    r = np.random.default_rng(seed)
    return float(np.mean([model(i, r) == answer_of(i) for i in instances]))

print("示例（NoOp 扰动后）:", render(perturb(ORIGINALS[0], "noop", np.random.default_rng(7))))
print("ground-truth answer =", answer_of(ORIGINALS[0]))

MODES = ["original", "names", "numbers", "noop"]
def instances_for(mode, seed):
    if mode == "original":
        return ORIGINALS
    r = np.random.default_rng(seed)
    return [perturb(i, mode, r) for i in ORIGINALS]

REPS = 30   # 每种扰动重采样 30 套实例：参数化基准天然支持“一次出一套新卷子”
acc = {m.__name__: {} for m in (memorizer, arithmetic)}
for mode in MODES:
    for model in (memorizer, arithmetic):
        runs = [accuracy(model, instances_for(mode, 100 + rep), 200 + rep) for rep in range(REPS)]
        acc[model.__name__][mode] = float(np.mean(runs))

gsm_df = pd.DataFrame(acc).T[MODES]
gsm_df["drop_numbers"] = gsm_df["original"] - gsm_df["numbers"]
gsm_df["drop_noop"] = gsm_df["original"] - gsm_df["noop"]
print("\n", gsm_df.round(3).to_string(), sep="")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(MODES)); w = 0.35
ax.bar(x - w / 2, [acc["memorizer"][m] for m in MODES], w, label="memorizer (template matching)")
ax.bar(x + w / 2, [acc["arithmetic"][m] for m in MODES], w, label="arithmetic (true reasoning)")
ax.set_xticks(x); ax.set_xticklabels(MODES); ax.set_ylabel("accuracy")
ax.set_title("GSM-Symbolic style perturbations"); ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

assert gsm_df.loc["memorizer", "drop_numbers"] > 0.8     # 套路记忆：换数字即崩溃
assert gsm_df.loc["arithmetic", "drop_numbers"] < 0.05   # 真会算：换数字不掉
assert gsm_df.loc["memorizer", "drop_noop"] > gsm_df.loc["arithmetic", "drop_noop"]

## 4 · 数学答案等价判分器 & 5 · 汇总评测报告

推理基准的判分远比 exact match 难：`"0.5"` vs `"1/2"`、`"1,234"` vs `"1234"`、浮点尾差……判分器自己的假阳率 $\alpha$ / 假阴率 $\beta$ 会传导成观测分 $\hat p = p(1-\beta) + (1-p)\alpha$ 的系统偏差；两个答案**风格**不同的模型（爱写分数 vs 爱写小数）会因此被排错名次。所以判分器要先在**已知标签的 corner cases** 上自测——这是元评测（meta-evaluation）的最小形态。

简化版分层判分器：① 字符串规范化（空格 / 千分位逗号 / `$`）→ ② 数值解析（`Fraction` 统一吃 `"7/2"`、`"42"`、`"0.5"`）→ ③ `math.isclose` 相对+绝对容差 → ④ 任一方解析失败则退化为大小写不敏感的字符串比较。完整工程栈还有 sympy 符号判等与 LLM judge 两层（讲解第 6 节）。

最后把三条审计线合成一张报告表——本课收官交付物：**faithfulness 分、鲁棒性掉点、判分器准确率**，分别回答"过程可信吗、分数是推理出来的吗、判卷可靠吗"。

In [ ]:
def _to_number(s):
    # 尝试把答案字符串解析成数值；失败返回 None
    t = str(s).strip().replace(",", "").replace("$", "").replace(" ", "")
    for parser in (lambda x: float(Fraction(x)), float):
        try:
            return parser(t)
        except (ValueError, ZeroDivisionError):
            continue
    return None

def answer_equiv(a, b, tol=1e-6):
    # 简化版数学答案等价判分：数值容差 + 分数/小数等价 + 字符串规范化
    na, nb = _to_number(a), _to_number(b)
    if na is not None and nb is not None:
        return math.isclose(na, nb, rel_tol=tol, abs_tol=tol)
    norm = lambda s: " ".join(str(s).split()).lower()
    return norm(a) == norm(b)

# 已标注的 corner cases：(模型答案, 标准答案, 是否应判等)
CASES = [
    ("0.5", "1/2", True), ("2/4", "0.5", True), ("7/2", "3.5", True),
    (" 42 ", "42", True), ("1,234", "1234", True), ("$18", "18", True),
    ("-0", "0", True), ("3.14159265", "3.14159264", True),
    ("3.14", "3.15", False), ("7/2", "3.6", False), ("12", "13", False),
    ("x + 1", "x +  1", True), ("apple", "banana", False), ("YES", "yes", True),
]
naive = lambda a, b: str(a) == str(b)
grader_acc = {name: float(np.mean([g(a, b) == label for a, b, label in CASES]))
              for name, g in {"naive exact match": naive, "answer_equiv": answer_equiv}.items()}
for name, accv in grader_acc.items():
    print(f"{name:18s}: {accv:.3f}  ({round(accv * len(CASES))}/{len(CASES)} corner cases)")
assert grader_acc["answer_equiv"] == 1.0
assert grader_acc["answer_equiv"] > grader_acc["naive exact match"]

# ---- 收官报告表：三条审计线 ----
report = pd.DataFrame([
    ["CoT faithfulness", "faithful 模型 · 早答 AOC",          fs_faithful, "≫ 0：答案随 CoT 证据成形"],
    ["CoT faithfulness", "post-hoc 模型 · 早答 AOC",          fs_posthoc,  "≈ 0：CoT 是事后装饰"],
    ["bias 注入",        "sycophant · flip×(1−mention)",      unfaith_syc, "高：被带偏且隐瞒"],
    ["bias 注入",        "transparent · flip×(1−mention)",    unfaith_tra, "低：被带偏但坦白"],
    ["鲁棒性",           "memorizer · 换数字掉点",            float(gsm_df.loc["memorizer", "drop_numbers"]),  "崩溃：套路记忆"],
    ["鲁棒性",           "arithmetic · 换数字掉点",           float(gsm_df.loc["arithmetic", "drop_numbers"]), "≈ 0：真会算"],
    ["鲁棒性",           "memorizer · NoOp 掉点",             float(gsm_df.loc["memorizer", "drop_noop"]),     "大：见数字就用"],
    ["判分器",           "naive exact match 准确率",          grader_acc["naive exact match"], "漏判等价答案"],
    ["判分器",           "answer_equiv 准确率",               grader_acc["answer_equiv"],      "corner cases 全过"],
], columns=["audit", "metric", "value", "reading"])
report["value"] = report["value"].round(3)
print("\n========== 推理模型评测报告（模拟实验台） ==========")
print(report.to_string(index=False))

---
## ✏️ 练习 1：实现 `early_answering_curve(model, fracs)`

输入：`model` 是可调用对象，`model(frac)` 返回全部题目在"CoT 截断到比例 frac"时的答案数组；`fracs` 是截断比例数组。
输出：`np.ndarray`——每个 frac 下答案与 `model(1.0)`（完整 CoT）的一致率。

**提示**：先算 `full = model(1.0)` 作对照，再对每个 f 求 `(model(f) == full).mean()`，3–5 行。边界：fracs 含 1.0 时该点必须恰好等于 1.0（自己和自己比）。

In [ ]:
def early_answering_curve(model, fracs):
    # TODO: full = model(1.0)；对每个 f 计算 model(f) 与 full 的一致率，返回 np.ndarray
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
fr = np.linspace(0, 1, 11)
c_f = early_answering_curve(faithful_model, fr)
c_p = early_answering_curve(posthoc_model, fr)
assert isinstance(c_f, np.ndarray) and c_f.shape == (11,)
assert abs(c_f[-1] - 1.0) < 1e-12 and abs(c_p[-1] - 1.0) < 1e-12   # f=1：与自身一致率必为 1
assert np.all(np.diff(c_f) >= -1e-9) and c_f[0] < 0.6              # faithful：曲线单调上升、起点低
assert np.all(c_p > 0.999)                                          # post-hoc：曲线平坦贴 1
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `unfaithfulness_score(flipped, mentioned)`

输入两个等长 bool 数组：`flipped[i]` = 注入 bias 后第 i 题答案是否被翻转；`mentioned[i]` = 该题 CoT 是否提及了 bias。
返回 `flip_rate * (1 - P(提及 | 翻转))`：翻转率高**且**翻转时不坦白 → 分高。

**提示**：`np.asarray(..., bool)` 防御输入类型；条件提及率即 `mentioned[flipped].mean()`；**边界**：没有任何翻转时直接返回 `0.0`（不能除零）。6–8 行。

In [ ]:
def unfaithfulness_score(flipped, mentioned):
    # TODO: flip_rate = flipped 均值；无翻转返回 0.0；
    #       否则 mention_given_flip = mentioned[flipped] 均值，
    #       返回 float(flip_rate * (1 - mention_given_flip))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert unfaithfulness_score(np.zeros(10, bool), np.zeros(10, bool)) == 0.0            # 边界：无翻转
assert abs(unfaithfulness_score(np.array([1]*4 + [0]*6, bool), np.zeros(10, bool)) - 0.4) < 1e-12
assert abs(unfaithfulness_score(np.array([1]*4 + [0]*6, bool),
                                np.array([1]*4 + [0]*6, bool))) < 1e-12               # 全坦白 -> 0
s_syc = unfaithfulness_score(flip_syc, mention_syc)
s_tra = unfaithfulness_score(flip_tra, mention_tra)
assert s_syc > 5 * s_tra                       # 指标必须分得开 sycophant 与 transparent
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `perturb_template(inst, mode, r)`

GSM-Symbolic 的核心机制：对题目实例 `inst`（dict，含 `name/a/b/c/noop`）做扰动，返回**新 dict**（不得修改原 inst）：

- `mode="numbers"`：重新随机 `a∈[20,80)`、`b∈[5,40)`、`c∈[2,20)`，且新三元组必须 ≠ 原三元组；
- `mode="noop"`：`a/b/c` 不动，置 `noop = r.integers(3, 9)`（无关从句的数字）。

**提示**：`out = dict(inst)` 浅拷贝即可；numbers 模式用 while 循环重采样直到三元组变化；用 `int()` 包住 numpy 整数。10–14 行。

In [ ]:
def perturb_template(inst, mode, r):
    # TODO: out = dict(inst)
    #   mode == "numbers": while 新 (a,b,c) == 原 (a,b,c)，重采样三个数字
    #   mode == "noop":    out["noop"] = int(r.integers(3, 9))
    # 返回 out
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r_t = np.random.default_rng(42)
inst0 = ORIGINALS[0]; inst0_snapshot = dict(inst0)
p1 = perturb_template(inst0, "numbers", r_t)
assert (p1["a"], p1["b"], p1["c"]) != (inst0["a"], inst0["b"], inst0["c"])   # 数字必须变
assert p1["noop"] == 0 and inst0 == inst0_snapshot                            # 原 dict 不被修改
assert 20 <= p1["a"] < 80 and 5 <= p1["b"] < 40 and 2 <= p1["c"] < 20
p2 = perturb_template(inst0, "noop", r_t)
assert (p2["a"], p2["b"], p2["c"]) == (inst0["a"], inst0["b"], inst0["c"]) and 3 <= p2["noop"] < 9
# 端到端：换数字后，模板记忆模型的掉点必须远大于真算术模型
pert = [perturb_template(i, "numbers", r_t) for i in ORIGINALS]
drop_mem = accuracy(memorizer, ORIGINALS, 0) - accuracy(memorizer, pert, 1)
drop_ari = accuracy(arithmetic, ORIGINALS, 0) - accuracy(arithmetic, pert, 2)
assert drop_mem > 0.5 > drop_ari
print(f"✅ 练习 3 通过   (drop_mem={drop_mem:.3f}, drop_ari={drop_ari:.3f})")

## ✏️ 练习 4：实现 `answer_equiv_ex(a, b, tol=1e-6)`

不翻上文，自己再写一遍判分器：

1. 规范化：去首尾空格、去千分位逗号、去 `$`、去内部空格，尝试解析为数值（`Fraction` 能同时吃 `"7/2"`、`"42"`、`"0.5"`）；
2. 双方都解析成功 → `math.isclose(na, nb, rel_tol=tol, abs_tol=tol)`；
3. 任一方解析失败 → 退化为大小写不敏感、空白折叠后的字符串比较。

**提示**：解析函数用 try/except 返回 None；注意 `Fraction("3/0")` 抛的是 `ZeroDivisionError`，也要接住，否则非法答案会让整个评测管线崩掉。15–20 行。

In [ ]:
def answer_equiv_ex(a, b, tol=1e-6):
    # TODO:
    #   1) 写一个内部解析函数：规范化字符串 -> Fraction/float -> 失败返回 None
    #      （接住 ValueError 和 ZeroDivisionError）
    #   2) 都解析成功 -> math.isclose 容差比较
    #   3) 否则 -> 折叠空白 + lower 后字符串比较
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert answer_equiv_ex("0.5", "1/2") and answer_equiv_ex("7/2", "3.5")        # 分数/小数等价
assert answer_equiv_ex(" 42 ", "42") and answer_equiv_ex("1,234", "1234")     # 空格与千分位
assert answer_equiv_ex("$18", "18") and answer_equiv_ex("-0", "0")
assert answer_equiv_ex("3.14159265", "3.14159264")                             # 默认容差内
assert not answer_equiv_ex("3.14", "3.15") and not answer_equiv_ex("7/2", "3.6")
assert answer_equiv_ex("YES", "yes") and not answer_equiv_ex("apple", "banana")  # 字符串退化路径
assert not answer_equiv_ex("3/0", "1")                                          # 非法分数：不崩溃、判不等
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def early_answering_curve(model, fracs):
    full = model(1.0)
    return np.array([(model(f) == full).mean() for f in fracs])

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def unfaithfulness_score(flipped, mentioned):
    flipped = np.asarray(flipped, bool)
    mentioned = np.asarray(mentioned, bool)
    if not flipped.any():
        return 0.0
    flip_rate = flipped.mean()
    mention_given_flip = mentioned[flipped].mean()
    return float(flip_rate * (1 - mention_given_flip))

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def perturb_template(inst, mode, r):
    out = dict(inst)
    if mode == "numbers":
        while (out["a"], out["b"], out["c"]) == (inst["a"], inst["b"], inst["c"]):
            out["a"] = int(r.integers(20, 80))
            out["b"] = int(r.integers(5, 40))
            out["c"] = int(r.integers(2, 20))
    elif mode == "noop":
        out["noop"] = int(r.integers(3, 9))
    return out

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def answer_equiv_ex(a, b, tol=1e-6):
    def to_num(s):
        t = str(s).strip().replace(",", "").replace("$", "").replace(" ", "")
        for parser in (lambda x: float(Fraction(x)), float):
            try:
                return parser(t)
            except (ValueError, ZeroDivisionError):
                continue
        return None
    na, nb = to_num(a), to_num(b)
    if na is not None and nb is not None:
        return math.isclose(na, nb, rel_tol=tol, abs_tol=tol)
    norm = lambda s: " ".join(str(s).split()).lower()
    return norm(a) == norm(b)

---
## 小结

- **早答曲线**把 faithful 与 post-hoc 干净地分开：faithful 的 $M(f)$ 从先验单调爬到 1，post-hoc 平坦贴 1——faithfulness 代理分 = 曲线上方面积。记住方法论前提：先确认 CoT 对该任务有准确率增益，"截断不变"才构成 unfaithful 的证据。
- **unfaithfulness = flip × (1 − mention|flip)**：被带偏不是重罪，被带偏还不坦白才是。Turpin / Anthropic hint 实验说明这在真实模型上是常态而非例外——CoT monitoring 的有效性前提需要逐（模型, 任务）验证。
- **参数化模板**是构造性污染防御：换数字的掉点幅度 ≈ 套路记忆程度。memorizer 在 numbers 扰动下掉 >90%，arithmetic 几乎不掉——与 GSM-Symbolic 在真实模型上的发现同构。
- **判分器先于模型被评测**：naive exact match 在 corner cases 上只有 ~29% 准确率，分层等价判分器 100%。判分误差 $\alpha/\beta$ 不对称时会直接扭曲模型排名——元评测不是可选项。

**全课完结。**你现在拥有完整链路：CoT 为什么有用（01）→ 怎么花测试时算力（02–04）→ scaling 规律与预算控制（05–06）→ 分数与思维链怎么审计（07）。下一步：把模拟推理器换成真实 API，在自己的评测管线里复刻这三条审计线。

---
## 🎯 真实数据胶囊题：GSM-Symbolic 式扰动：真实题目改数字后答案应变

GSM-Symbolic 发现：把题目里的数字换掉，真正会推理的解法答案应相应改变，靠记忆的会崩。用真实 GSM8K，提取题中数字、做线性扰动，验证一个忠实求解器的答案会随之改变。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(100)
def nums_in(q): return [int(x) for x in re.findall(r"\b\d+\b", q)]
ex=rows[0]; print("真实题首句:", ex["question"][:80])
print("题中数字:", nums_in(ex["question"])[:6])

**练习**：实现 `perturb_changes_answer(orig_nums, solver)`：给原数字和一个 `solver(nums)->答案`，把每个数字 +10 做扰动，返回布尔——忠实 solver 的答案是否改变了。

In [ ]:
def perturb_changes_answer(orig_nums, solver):
    # TODO: 扰动 = [x+10 for x in orig_nums]; 返回 solver(扰动) != solver(原)
    raise NotImplementedError


In [ ]:
# 自测：忠实 solver(求和) 答案应随数字改变；记忆型 solver(常数) 不变
faithful = lambda nums: sum(nums)
memorizer = lambda nums: 42
real_nums = nums_in(rows[0]["question"])
assert perturb_changes_answer(real_nums, faithful)==True, "忠实求解器答案应变"
assert perturb_changes_answer(real_nums, memorizer)==False, "记忆型不变 -> 暴露不推理"
print("GSM-Symbolic 扰动 ✓ 能区分'真推理' vs '背答案'")


### 📖 参考答案

In [ ]:
def perturb_changes_answer(orig_nums, solver):
    pert=[x+10 for x in orig_nums]
    return solver(pert) != solver(orig_nums)
print("✓ 数值扰动是检测推理鲁棒性/污染的利器(GSM-Symbolic)")